# Regulatory Corpus Cleaning Pipeline (Colab)
===========================================

Cleans OCR artifacts and formatting issues from Indonesian regulatory documents
for transformer-based semantic analysis.

**Runtime**: Python 3 with GPU (optional)
**Output**: Cleaned corpus in `/content/cleaned/`

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

#@title Set paths
CORPUS_DIR = '/content/drive/MyDrive/research/regulatory_corpus'  #@param {type:"string"}
OUTPUT_DIR = '/content/cleaned'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Corpus directory: {CORPUS_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy

## 2. Define Cleaning Functions

In [ ]:
import re

# ============================================================================
# OCR FIX DICTIONARY
# ============================================================================
OCR_FIXES = {
    # Context-dependent fixes
    'KECERDASAN INDONESIA': 'KECERDASAN ARTIFISIAL',
    
    # Common OCR confusions
    'tr{I': 'dan',
    'tr{ir': 'dan',
    'tr{rr': 'dan',
    'NEPLTBLIK': 'REPUBLIK',
    'NEPUELIK': 'REPUBLIK',
    'NEPUBUK': 'REPUBLIK',
    'REPIJBUK': 'REPUBLIK',
    'REPIITEUK': 'REPUBLIK',
}

# Page/line markers to remove
PAGE_MARKER_PATTERN = re.compile(r'SK No \d+[A-Z]+')
LINE_NUMBER_PATTERN = re.compile(r'^\s*\d+\s*\|', re.MULTILINE)


def remove_page_artifacts(text):
    """Remove page numbers, section markers, and line numbers"""
    text = PAGE_MARKER_PATTERN.sub('', text)
    text = LINE_NUMBER_PATTERN.sub('', text)
    text = re.sub(r'\d+\s+BAB\s+\d+', '', text)
    return text


def fix_ocr_errors(text):
    """Apply dictionary-based OCR fixes"""
    for wrong, correct in OCR_FIXES.items():
        text = text.replace(wrong, correct)
    return text


def normalize_whitespace(text):
    """Normalize spacing and line breaks"""
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\s+([.,;:])', r'\1', text)
    text = re.sub(r'([.,;:])([A-Z])', r'\1 \2', text)
    return text


def fix_sentence_boundaries(text):
    """Fix broken sentence boundaries from OCR line breaks"""
    text = re.sub(r'\.\s+\.', '.', text)
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    return text


def clean_regulatory_document(text):
    """Main cleaning function - applies all cleaning steps"""
    text = remove_page_artifacts(text)
    text = fix_ocr_errors(text)
    text = normalize_whitespace(text)
    text = fix_sentence_boundaries(text)
    return text

## 3. Process All Files

In [ ]:
import os

# Find all txt files
files = [f for f in os.listdir(CORPUS_DIR) if f.endswith('.txt')]

print(f"Found {len(files)} files to process:\n")

for filename in files:
    filepath = os.path.join(CORPUS_DIR, filename)
    print(f"Processing: {filename}")
    
    # Read file
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    original_length = len(content)
    
    # Clean content
    cleaned = clean_regulatory_document(content)
    
    cleaned_length = len(cleaned)
    
    # Save cleaned version
    output_path = os.path.join(OUTPUT_DIR, filename)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(cleaned)
    
    print(f"  Original: {original_length:,} chars")
    print(f"  Cleaned:  {cleaned_length:,} chars")
    print(f"  Removed:  {original_length - cleaned_length:,} chars")
    print(f"  Saved to: {output_path}\n")

print("✓ Cleaning complete!")

## 4. Verify Cleaned Output

In [ ]:
# Sample verification
for filename in os.listdir(OUTPUT_DIR):
    if filename.endswith('.txt'):
        filepath = os.path.join(OUTPUT_DIR, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            sample = f.read()[:500]
        print(f"=== {filename} (first 500 chars) ===\n")
        print(sample)
        print("\n" + "="*50 + "\n")